In [12]:
import os
import numpy as np
import pandas as pd

SEED = 9
BOOTSTRAP_REPS = 200 + (10 * SEED)
NUM_FEATURES = 5

In [ ]:
data_file = "secom.data"
labels_file = "secom_labels.data"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

robustness_csv_path = os.path.join(output_dir, "robustness_comparison.csv")
sensitivity_txt_path = os.path.join(output_dir, "sensitivity_report.txt")

df_data = pd.read_csv(data_file, header=None, sep=r"\s+")
labels_df = pd.read_csv(labels_file, header=None, sep=r"\s+")
df = df_data.copy()
df["Outcome"] = labels_df.iloc[:, 0].values

numeric_cols = df.columns[:-1]
valid_cols = []
for col in numeric_cols:
    pass_count = df[df["Outcome"] == -1][col].dropna().count()
    fail_count = df[df["Outcome"] == 1][col].dropna().count()
    if pass_count >= 10 and fail_count >= 10:
        valid_cols.append(col)
         

np.random.seed(SEED)
selected_features = np.random.choice(
     valid_cols, size=min(NUM_FEATURES, len(valid_cols)), replace=False
)

pass_full = df[df["Outcome"] == -1]
fail_full = df[df["Outcome"] == 1]

def compute_strategy_metrics(use_imputations=False):
    work_df = df.copy()
    if use_imputations:
        for col in selected_features:
            median_val = work_df[col].median()
            work_df[col] = work_df[col].fillna(median_val)

    p_group = work_df[work_df["Outcome"] == -1]
    f_group = work_df[work_df["Outcome"] == 1]

    mean_diffs = {}
    boot_diffs = {col: [] for col in selected_features}

    for col in selected_features:
        p_vals = p_group[col].dropna() if not use_imputations else p_group[col]
        f_vals = f_group[col].dropna() if not use_imputations else f_group[col]

        mean_diffs[col] = f_vals.mean() - p_vals.mean()

        for _ in range(BOOTSTRAP_REPS):
            p_sample = p_vals.sample(n=len(p_vals), replace=True, random_state=None)
            f_sample = f_vals.sample(n=len(f_vals), replace=True, random_state=None)
            boot_diffs[col].append(f_sample.mean() - p_sample.mean())

        return mean_diffs, boot_diffs

mean_diffs_ac, boot_diffs_ac = compute_strategy_metrics(use_imputations=False)
mean_diffs_imp, boot_diffs_imp = compute_strategy_metrics(use_imputations=True)

comparison_data = []
changed_sign_count = 0
max_mean_diff_change = 0.0

def compute_strategy_metrics(use_imputations=False):
    work_df = df.copy()
    if use_imputations:
        for col in selected_features:
            work_df[col] = work_df[col].fillna(work_df[col].median())

    p_group = work_df[work_df["Outcome"] == -1]
    f_group = work_df[work_df["Outcome"] == 1]

    mean_diffs = {}
    boot_diffs = {col: [] for col in selected_features}

    for col in selected_features:
        p_vals = p_group[col].dropna()
        f_vals = f_group[col].dropna()

        mean_diffs[col] = f_vals.mean() - p_vals.mean()

        for _ in range(BOOTSTRAP_REPS):
            p_sample = p_vals.sample(n=len(p_vals), replace=True)
            f_sample = f_vals.sample(n=len(f_vals), replace=True)
            boot_diffs[col].append(f_sample.mean() - p_sample.mean())

    
    return mean_diffs, boot_diffs
# Recompute both strategies using the final, correctly indented function definition.
mean_diffs_ac, boot_diffs_ac = compute_strategy_metrics(use_imputations=False)
mean_diffs_imp, boot_diffs_imp = compute_strategy_metrics(use_imputations=True)

ranking_ac = sorted(
    selected_features,
    key=lambda c: abs(mean_diffs_ac[c]),
    reverse=True,
)
ranking_imp = sorted(
    selected_features,
    key=lambda c: abs(mean_diffs_imp[c]),
    reverse=True,
)
changed_ranking_features = []

for idx, col in enumerate(selected_features):
    diff_ac = mean_diffs_ac[col]
    diff_imp = mean_diffs_imp[col]

    ci_ac = np.percentile(boot_diffs_ac[col], [2.5, 97.5])
    ci_imp = np.percentile(boot_diffs_imp[col], [2.5, 97.5])

    diff_change = abs(diff_imp - diff_ac)
    if diff_change > max_mean_diff_change:
        max_mean_diff_change = diff_change

    sign_changed = np.sign(diff_ac) != np.sign(diff_imp)
    if sign_changed:
        changed_sign_count += 1

    rank_ac = ranking_ac.index(col)
    rank_imp = ranking_imp.index(col)
    if rank_ac != rank_imp:
        changed_ranking_features.append(col)

    comparison_data.append({
        "Feature": col,
        "AC_Mean_Diff": diff_ac,
        "AC_CI_Lower": ci_ac[0],
        "AC_CI_Upper": ci_ac[1],
        "Imp_Mean_Diff": diff_imp,
        "Imp_CI_Lower": ci_imp[0],
        "Imp_CI_Upper": ci_imp[1],
        "Sign_Changed": sign_changed,
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv(robustness_csv_path, index=False)

with open(sensitivity_txt_path, mode="w", encoding="utf-8") as rep:
    rep.write("STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n")
    rep.write("=" * 50 + "\n")
    rep.write(f"Bootstrap Repetitions: {BOOTSTRAP_REPS}\n")
    rep.write(f"Selected Features: {list(selected_features)}\n")
    rep.write(f"Largest Change in Mean Difference: {max_mean_diff_change:.4f}\n")
    rep.write(f"Features with Changed Difference Sign: {changed_sign_count}\n")
    rep.write(
        
    )


KeyError: np.int64(574)